# Modelling a credit default swap index (CDX) and handling instrument events

This notebook demonstrates the concepts described in the following KBs:

* Mastering a `CdsIndex` instrument, establishing a position, and performing a valuation: https://support.lusid.com/docs/modelling-cds-index-instruments-in-lusid
* Handling instrument events: https://support.lusid.com/docs/handling-premium-and-credit-events-for-cdx-instruments

All the information also applies to a standard (non-index) `CreditDefaultSwap` instrument unless otherwise stated.

* Start date: 20 March 2024
* Maturity date: 20 June 202
* Transaction/settlement date: 20 September 2024
* 5% premium ('coupon') payment
* Premiums payable every twelve months (in arrears) after purchase - 20 June and 20 December (so first payment due on 20 December 2024)
* A `CdxCreditEvent` must be manually loaded into a corporate action source to trigger a payout if a constituent defaults.

## Setup

In [ ]:
import os
import pandas as pd
import numpy as np
import json
import uuid
from IPython.core.display import HTML
import logging
from datetime import datetime, timezone, timedelta
logging.basicConfig(level = logging.ERROR)

import finbourne.sdk.services.lusid.api as la
import finbourne.sdk.services.lusid.models as lm

from finbourne.sdk.extensions import SyncApiClientFactory, RefreshingToken
from finbourne.sdk.exceptions import ApiException
from finbourne_sdk_utils.pandas_utils.lusid_pandas import lusid_response_to_data_frame
from finbourne_sdk_utils.lpt.lpt import to_date

import fbnconfig

# Set pandas display options
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.options.display.float_format = "{:,.2f}".format

# Authenticate to SDK
# Run the Notebook in Jupyterhub for your LUSID domain and authenticate automatically
secrets_path = os.getenv("FBN_SECRETS_PATH")
# Run the Notebook locally using a secrets file (see https://support.lusid.com/docs/how-do-i-use-an-api-access-token-with-the-lusid-sdk)
if secrets_path is None:
    secrets_path = os.path.join(os.path.dirname(os.getcwd()), "secrets.json")

# Initiate an API Factory which is the client side object for interacting with LUSID APIs
api_factory = SyncApiClientFactory(
    access_token=RefreshingToken(),
    secrets_path=secrets_path,
    app_name="LusidJupyterNotebook"
)
    
# Confirm success by printing SDK version
api_url = api_factory.get_api_configuration().host.replace("api", "")
print ('LUSID Environment :', api_url)
display(pd.DataFrame(api_factory.build(la.ApplicationMetadataApi).get_lusid_versions().to_dict()))

In [ ]:
# Build all the required APIs
try:
    instruments_api = api_factory.build(la.InstrumentsApi)
    instrument_events_api = api_factory.build(la.InstrumentEventsApi)
    corporate_action_sources_api = api_factory.build(la.CorporateActionSourcesApi)
    aggregation_api = api_factory.build(la.AggregationApi)
    recipe_api = api_factory.build(la.ConfigurationRecipeApi)
    quotes_api = api_factory.build(la.QuotesApi)
    property_definition_api = api_factory.build(la.PropertyDefinitionsApi)
    transaction_portfolios_api = api_factory.build(la.TransactionPortfoliosApi)
    portfolios_api = api_factory.build(la.PortfoliosApi)
    abor_api = api_factory.build(la.AborApi)
    abor_config_api = api_factory.build(la.AborConfigurationApi)
    chart_accts_api = api_factory.build(la.ChartOfAccountsApi)
    transaction_config_api = api_factory.build(la.TransactionConfigurationApi)
    fund_api = api_factory.build(la.FundsApi)
    feetypes_api = api_factory.build(la.FeeTypesApi)
    fundconfig_api = api_factory.build(la.FundConfigurationApi)
    transaction_fees_api = api_factory.build(la.TransactionFeeTypesApi)
    print("All APIs built correctly")
except ApiException as e:
    print(e)

## Create a scope and code for entities in the Notebook

Keep data segregated from other data in LUSID.

In [ ]:
module_scope = "FBNTutorials"
module_code = "CDX-CDS-Events"
print(f"'{module_scope}\\{module_code}' scope and code created.")

## Create property types

Create SHKs to split out premium and credit cash payments from other cash balances in holding reports.

In [ ]:
def create_property_type(property_domain, property_scope, property_code, data_type):
    property_type_request = lm.CreatePropertyDefinitionRequest(
        domain = property_domain,
        scope = property_scope,
        code = property_code,
        display_name = property_code,
        data_type_id = lm.ResourceId(scope = "system", code = data_type)
    )

    try:
        property_type_response = property_definition_api.create_property_definition(
            create_property_definition_request = property_type_request
        )
        print(f"Property type created with the following key: {property_type_response.key}")
        return property_type_response.key
    except ApiException as e:
        if json.loads(e.body)["name"] == "PropertyAlreadyExists":
            logging.info(
                f"Property type with the following key already exists: {property_type_request.domain}/{property_type_request.scope}/{property_type_request.code}"
            )  
        return f"{property_type_request.domain}/{property_type_request.scope}/{property_type_request.code}"

In [ ]:
premium_shk = create_property_type("Transaction", "SHKs", "CDXPremiumPayments", "string")
credit_shk = create_property_type("Transaction", "SHKs", "CDXProtectionPayouts", "string")

## Create transaction types and sides

Required for both purchase transactions and for transactions automatically generated by instrument events.

All created in a custom transaction type scope, which must be registered with the portfolio in which transactions are loaded.

In [ ]:
def check_TT(tt, scope):
    try:
        tt_response = transaction_config_api.get_transaction_type(source = f"default", type = tt, scope=scope)
        print(f"\n{tt} transaction type:")
        display(lusid_response_to_data_frame(tt_response.aliases))
        display(lusid_response_to_data_frame(tt_response.movements))
        display(lusid_response_to_data_frame(tt_response.calculations))
    except ApiException as e:
        print(e)
        
def check_side(side, scope):
    try:
        side_response = transaction_config_api.get_side_definition(scope = scope, side = side)
        print(f"\n{side} side:")
        side_response_df = lusid_response_to_data_frame(side_response).transpose()
        side_response_df.drop(side_response_df.filter(regex='links').columns, axis=1, inplace=True)
        display(side_response_df)  
    except ApiException as e:
        print(e)

In [ ]:
with open(f"standard-txn-types.{module_scope}-{module_code}.json") as f:
    deployment = fbnconfig.undump_deployment(json.load(f))

# Source the access token from the FBN_ACCESS_TOKEN env var, falling back to
# the contents of the file named by FBN_ACCESS_TOKEN_FILE
access_token = os.environ.get("FBN_ACCESS_TOKEN")
if access_token is None:
    with open(os.environ["FBN_ACCESS_TOKEN_FILE"]) as f:
        access_token = f.read().strip()

actions = fbnconfig.deploy(
    deployment,
    os.environ["FBN_BASE_URL"],
    access_token,
    write_control=0,  # 1=upsert, 2=no-update, 4=no-delete, 8=delete-if-owned
)
actions

In [ ]:
check_side("Side1", f"{module_scope}-{module_code}")
check_side("Side2", f"{module_scope}-{module_code}")

check_TT("BuyProtection", f"{module_scope}-{module_code}")
check_side("Protection", f"{module_scope}-{module_code}")
check_side("Accrued", f"{module_scope}-{module_code}")

check_TT("CreditPremiumCashFlow", f"{module_scope}-{module_code}")
check_side("SignedSide1", f"{module_scope}-{module_code}")

check_TT("ProtectionPayoutCashFlow", f"{module_scope}-{module_code}")
check_TT("Maturity", f"{module_scope}-{module_code}")

## Master instruments

In [ ]:
def master_instrument(id, currency, couponday, startdate, maturitydate, premium):
    
    instrument_request = {
        id: lm.InstrumentDefinition(
            name = id,
            identifiers = {"ClientInternal": lm.InstrumentIdValue(value = id.replace(" ", ""))},
            definition = lm.CdsIndex(
                instrument_type = "CdsIndex", 
                start_date = datetime.strptime(startdate, '%Y-%m-%d').replace(tzinfo=timezone.utc).isoformat(),
                maturity_date = datetime.strptime(maturitydate, '%Y-%m-%d').replace(tzinfo=timezone.utc).isoformat(),
                # Note that a CdsIndex instrument is unitised and the quantity specified in transactions
                notional = 1,
                # Specified as a number rather than percentage, eg. so a premium of 0.375% is specified as 0.00375
                coupon_rate = premium / 100,
                identifiers={},
                flow_conventions = lm.CdsFlowConventions(
                    currency = currency,
                    # Premiums are paid twice yearly
                    payment_frequency = "6M", # Must be 3M or 6M
                    day_count_convention = "Actual360",
                    roll_convention = f"{couponday}",
                    business_day_convention = "F",
                    payment_calendars = ["EUR"],
                    # settleDays=0,
                    # resetDays=0,
                    reset_calendars = ["EUR"]
                )
            )
        )
    }
    
    try:
        instrument_response = instruments_api.upsert_instruments(
            request_body = instrument_request,
            scope = f"{module_scope}{module_code}"
        )
        #print(instrument_response)
        # Return LUID from (only) Instrument object
        return list(instrument_response.values.values())[0].lusid_instrument_id 
    except ApiException as e:
        print(e)

In [ ]:
luid_dict = {}
luid_dict["MarkitiTraxxCDX"] = master_instrument("Markit iTraxx Crossover Series 41 5Y", "EUR", "IMM", "2024-03-20", "2029-06-20", 5)

for k, v in luid_dict.items():
    print(f"{k}: {v}")

In [ ]:
def list_instrs():
    instr_response = instruments_api.list_instruments(scope=f"{module_scope}{module_code}")
    instr_response_df = lusid_response_to_data_frame(instr_response, use_camel_case=True)
    instr_response_df.drop(instr_response_df.filter(regex='version|href|staged').columns, axis=1, inplace=True)
    display(instr_response_df.transpose())

list_instrs()

## Create a recipe

Must be specified as a portfolio recipe to enable instrument events. Can also be used as a valuation recipe.

Note the pricing model has been changed to the dedicated, recommended `CdsLookupPricer`.

In [ ]:
recipe = lm.ConfigurationRecipe(
    # Put the recipe in the same scope as the portfolio
    scope = module_scope,
    # Give the recipe a unique code in the scope
    code = f"{module_code}-CdsLookupPricer",
    description = "A recipe to value a CdsIndex",
    market = lm.MarketContext(
        market_rules = [
            # Look up FX spot rates in the LUSID quote store, if needed
            lm.MarketDataKeyRule(
                key = "Fx.CurrencyPair.*",
                supplier = "Lusid",
                data_scope = f"{module_scope}{module_code}",
                quote_type = "Rate",
                field = "mid",
                quote_interval = "1D.0D",
            ),
            lm.MarketDataKeyRule(
                key = "Quote.LusidInstrumentId.*",
                supplier = "Lusid",
                data_scope = f"{module_scope}{module_code}",
                quote_type = "Price",
                field = "mid",
                quote_interval = "1D.0D",
            )
        ]
    ),
    # Change pricing model      
    pricing=lm.PricingContext(
        model_rules=[
            lm.VendorModelRule(
                supplier="Lusid",
                model_name="CdsLookupPricer",
                instrument_type="CdsIndex"
            )
        ],
        # LUSID implicitly creates protection and payment legs but the CdsLookupPricer pricing model does not yet value legs separately
        options=lm.PricingOptions(
            produce_separate_result_for_linear_otc_legs=True
        )
    )
)

try:
    recipe_api.upsert_configuration_recipe(
        upsert_recipe_request = lm.UpsertRecipeRequest(
            configuration_recipe = recipe
        )
    )
    print("Success")
except ApiException as e:
    print(e)

In [ ]:
# Confirm recipe upsert and show the many options that are automatically set to default values by LUSID.
#config_recipe = recipe_api.list_configuration_recipes(filter=f"value.scope eq '{module_scope}' and value.code startswith '{module_code}'")
#config_recipe_df = lusid_response_to_data_frame(config_recipe)
#display(config_recipe_df.transpose())

## Create a corporate action source

Required if `CdxCreditEvent` (or `CdsCreditEvent` for a `CreditDefaultSwap`) is to be loaded in as a manual event, to trigger a protection payout.

In [ ]:
ca_source_definition = lm.CreateCorporateActionSourceRequest(
    scope=module_scope,
    code=module_code,
    display_name=f"{module_scope}/{module_code} CAS",
    instrument_scopes = [f"{module_scope}{module_code}"]
)

try:
    corporate_action_sources_api.create_corporate_action_source(
        create_corporate_action_source_request = ca_source_definition
    )
    print(f"{module_scope}/{module_code} CAS created")
except ApiException as e:
    print(e)

## Set up two EUR transaction portfolios

In one, the `CdsIndex` is purchased above par; in the other, below par - to demonstrate the impact on cost. See https://support.lusid.com/docs/modelling-cds-index-instruments-in-lusid#confirming-positions-on-the-settlement-date.

The portfolio recipe is set to the recipe created above to enable instrument events, and the transaction type scope, corporate action source and sub-holding keys are registered.

In [ ]:
def create_portfolio(name):
    portfolio_request=lm.CreateTransactionPortfolioRequest(
        display_name = f"{name} CDX portfolio",
        code = f"{module_code}-{name}",
        # Set the portfolio currency
        base_currency = "EUR",
        # Must be before first transaction recorded
        created = datetime.strptime("2024-01-01", '%Y-%m-%d').replace(tzinfo=timezone.utc).isoformat(),
        # Attempt to resolve transactions to instruments in the custom scope before falling back to the default scope
        instrument_scopes = [f"{module_scope}{module_code}"],
        # Register SHKs
        sub_holding_keys = [f"{premium_shk}", f"{credit_shk}"],
        # Register transaction type scope
        transactionTypeScope=f"{module_scope}-{module_code}",
        # Register portfolio recipe        
        instrumentEventConfiguration=lm.InstrumentEventConfiguration(
            recipeId=lm.ResourceId(
                scope=module_scope,
                code=f"{module_code}-CdsLookupPricer"
            )
        ),
        # Register corporate action source
        corporate_action_source_id=lm.ResourceId(
            scope=module_scope,
            code=module_code
        )
    )

    try:
        portfolio_response=transaction_portfolios_api.create_portfolio(
            scope = module_scope,
            create_transaction_portfolio_request = portfolio_request
        )
        print(f"Portfolio with display name '{portfolio_response.display_name}' created effective {str(portfolio_response.created)}")
    except ApiException as e:
        print(e)

In [ ]:
ports = ["AbovePar", "BelowPar"]
#ports = ["AbovePar"]
for port in ports:
    create_portfolio(port)

### Confirm portfolio details

In [ ]:
def get_port_details(port):
    portfolio_response = transaction_portfolios_api.get_details(scope = module_scope, code = f"{module_code}-{port}")
    portfolio_response_df = lusid_response_to_data_frame(portfolio_response).transpose()
    # Drop some noisy columns
    portfolio_response_df.drop(portfolio_response_df.filter(regex='version|href|staged|links|settlement').columns, axis=1, inplace=True)
    display(portfolio_response_df.transpose())
    
for port in ports:
    get_port_details(port)

### Load transactions into portfolios

LUSID converts a par-like `transactionPrice.price` (eg. `102`) to a rate (eg. `-0.02`). Note the `totalConsideration.amount` is set to `0` to trigger LUSID to calculate amounts (accrued interest and gross and total consideration) automatically. 

For more information, see https://support.lusid.com/docs/modelling-cds-index-instruments-in-lusid#booking-a-transaction-to-establish-a-position.

In [ ]:
# def create_transactions(port, txnid, tttype, luid, tradedate, settledate, quantity, price, ccy, totalcons, fees, interest):
def create_transactions(port, txnid, tttype, luid, tradedate, settledate, quantity, price, ccy):
    create_txn_request = {
        "number_one": lm.TransactionRequest(
            transaction_id=txnid,
            type=tttype,
            instrument_identifiers = {"Instrument/default/LusidInstrumentId": luid},
            transaction_date=datetime.strptime(tradedate, '%Y-%m-%d %H:%M:%S').replace(tzinfo=timezone.utc).isoformat(),
            settlement_date=datetime.strptime(settledate, '%Y-%m-%d %H:%M:%S').replace(tzinfo=timezone.utc).isoformat(),
            units=quantity,
            # This is the market price, used for gross consideration calculations
            transaction_price=lm.TransactionPrice(
                price=price, type="Price"
            ),
            total_consideration = lm.CurrencyAndAmount(
                currency = ccy,
                amount = 0
            )
        )
    }
    
    try:
        create_txn_response = transaction_portfolios_api.batch_upsert_transactions(
            scope = f"{module_scope}",
            code = f"{module_code}-{port}",
            success_mode="Partial",
            request_body = create_txn_request
        )
        print(create_txn_response.failed) if create_txn_response.failed else print("Success")
    except ApiException as e:
        print(e)

In [ ]:
# Above par-like price of 102, so cash and accrued interest received
# create_transactions("AbovePar", "Txn01", "BuyProtection", luid_dict["MarkitiTraxxCDX"], "2024-09-20 00:00:00", "2024-09-25 00:00:00", 100000, 102, "EUR", -2000, 0, -1270)
create_transactions("AbovePar", "Txn01", "BuyProtection", luid_dict["MarkitiTraxxCDX"], "2024-09-20 00:00:00", "2024-09-20 00:00:00", 100000, 102, "EUR")

# Below par-like price of 98, so accrued interest received but cash paid
# create_transactions("BelowPar", "Txn01", "BuyProtection", luid_dict["MarkitiTraxxCDX"], "2024-09-20 00:00:00", "2024-09-25 00:00:00", 100000, 98, "EUR", 2000, 0, -1270)
create_transactions("BelowPar", "Txn01", "BuyProtection", luid_dict["MarkitiTraxxCDX"], "2024-09-20 00:00:00", "2024-09-20 00:00:00", 100000, 98, "EUR")

### Confirm positions and audit output transactions

In [ ]:
def get_portfolio_holdings(port, date):      
    if date == "today":
        date = str(datetime.now().replace(microsecond=0))
    
    try:
        get_holdings_response = transaction_portfolios_api.get_holdings(
            scope = module_scope, 
            code = f"{module_code}-{port}",
            effective_at = datetime.strptime(date, '%Y-%m-%d %H:%M:%S').replace(tzinfo=timezone.utc).isoformat(),
            property_keys=["Instrument/default/Name"]
        )
        get_holdings_response_df = lusid_response_to_data_frame(get_holdings_response)
        get_holdings_response_df.rename(columns = {
            "sub_holding_keys.Transaction/SHKs/BondCollateralCoupons.value.label_value": "SHK",
            "sub_holding_keys.Transaction/SHKs/CDXProtectionPayouts.value.label_value": "ProtectionPayoutSHK",
            "sub_holding_keys.Transaction/SHKs/CDXPremiumPayments.value.label_value": "PremiumPaymentsSHK",
            "properties.Instrument/default/Name.value.label_value": "instrument"}, inplace = True)        
        # Drop some noisy columns
        get_holdings_response_df.drop(get_holdings_response_df.filter(regex='properties|sub_holding_keys').columns, axis=1, inplace=True)
        display(get_holdings_response_df)
    except ApiException as e:
        print(e)

In [ ]:
def get_output_transactions(port, start, end, all_data):
    try:
        output_transactions_response = transaction_portfolios_api.build_transactions(
            scope = module_scope, 
            code = f"{module_code}-{port}",
            transaction_query_parameters = lm.TransactionQueryParameters(
                start_date = datetime.strptime(start, '%Y-%m-%d').replace(tzinfo=timezone.utc).isoformat(),
                end_date = datetime.strptime(end, '%Y-%m-%d').replace(tzinfo=timezone.utc).isoformat()
            )
        )
        output_transactions_response_df = lusid_response_to_data_frame(output_transactions_response, use_camel_case=True)
        if all_data == False:
            output_transactions_response_df = output_transactions_response_df[[
                "transactionId", 
                "grossTransactionAmount", 
                "transactionAmount", 
                "properties.Transaction/default/BondInterest.value.metricValue.value", 
                "properties.Transaction/default/GrossConsideration.value.metricValue.value", 
                "totalConsideration.amount"
            ]]
            output_transactions_response_df.rename(columns = {
                "properties.Transaction/default/BondInterest.value.metricValue.value": "Transaction/default/BondInterest",
                "properties.Transaction/default/GrossConsideration.value.metricValue.value": "Transaction/default/GrossConsideration"}, inplace = True)        
            display(output_transactions_response_df)
        else:
            display(output_transactions_response_df.transpose())
    except ApiException as e:
        print(e)

In [ ]:
for port in ports:
    print(f"\n{port}")
    get_portfolio_holdings(port, "2024-09-25 23:59:59")  # Settlement date

In [ ]:
for port in ports:
    print(f"\n{port} purchase summary")
    get_output_transactions(port, "2024-01-01", "2024-09-25", False)

for port in ports:
    print(f"\n{port} purchase detail")
    get_output_transactions(port, "2024-01-01", "2024-09-25", True)

## Instrument events

### Examine state before triggering `ProtectionPayoutCashFlowEvent`

LUSID automatically generates a `CreditPremiumCashFlowEvent` on each premium payment date, and `MaturityEvent` on the maturity date.

Events are demonstrated on the `AbovePar` portfolio only (to reduce noise, since instrument events are the same for both).

In [ ]:
def query_instr_events(port):
    
    query_id_request = lm.QueryApplicableInstrumentEventsRequest(
        window_start = datetime.strptime("2015-01-01", '%Y-%m-%d').replace(tzinfo=timezone.utc).isoformat(),
        window_end = datetime.strptime("2030-01-01", '%Y-%m-%d').replace(tzinfo=timezone.utc).isoformat(),
        effective_at = datetime.strptime("2025-03-31", '%Y-%m-%d').replace(tzinfo=timezone.utc).isoformat(),
        portfolio_entity_ids = [
            lm.PortfolioEntityId(
                scope = module_scope,
                code = f"{module_code}-{port}",
            )
        ],
        forecasting_recipe_id = lm.ResourceId(
            scope = module_scope,
            code = f"{module_code}-CdsLookupPricer"
        )
    )
    
    try:
        query_id_response = instrument_events_api.query_applicable_instrument_events(
            query_applicable_instrument_events_request = query_id_request,
            limit = 200
        )
        display(lusid_response_to_data_frame(query_id_response, use_camel_case=True).transpose())
    except ApiException as e:
        print(e)

In [ ]:
# Just for AbovePar portfolio, since events are the same for both
print(ports[0])
query_instr_events(ports[0])

In [ ]:
# Just for AbovePar portfolio, since events are the same for both
pd.set_option('display.max_colwidth', 200)  
print(ports[0])
get_output_transactions(ports[0], "2024-01-01", "2030-01-01", False)

### Load `CdxCreditEvent` to trigger `ProtectionPayoutCashFlowEvent`

Note `ProtectionPayoutCashFlowEvent` is not loaded directly itself but rather triggered by loading `CdxCreditEvent` (`CdsCreditEvent` for a `CreditDefaultSwap`) into the corporate action source registered with the portfolio. 

See https://support.lusid.com/docs/handling-premium-and-credit-events-for-cdx-instruments#handling-credit-events.

In [ ]:
def load_event_into_CAS(luid, rate, weight):
    
    ca_request = lm.UpsertInstrumentEventRequest(
        # An event must have a unique identifier within a corporate action source
        instrument_event_id = f"{luid}",
        instrument_identifiers = {"Instrument/default/LusidInstrumentId": luid },
        description="Manually-loaded CdxCreditEvent",
        instrument_event = lm.CdxCreditEvent(
            effective_date=datetime.strptime("2025-02-15", '%Y-%m-%d').replace(tzinfo=timezone.utc).isoformat(),
            auction_date=datetime.strptime("2025-03-20", '%Y-%m-%d').replace(tzinfo=timezone.utc).isoformat(),
            payment_date = datetime.strptime("2025-03-25", '%Y-%m-%d').replace(tzinfo=timezone.utc).isoformat(),
            recovery_rate=rate,
            constituent_weight=weight,
            constituent_reference='Intram',
            instrument_event_type="CdxCreditEvent"
        ),
    )
    
    # Upsert instrument events as corporate action
    try:
        ca_response = corporate_action_sources_api.upsert_instrument_events(
            scope = module_scope,
            code = module_code,
            upsert_instrument_event_request = [ca_request]
        )
        print("Success")
    except ApiException as e:
        print(e)

In [ ]:
load_event_into_CAS(luid_dict["MarkitiTraxxCDX"], 0.76, 0.01333)

In [ ]:
def check_CAS_event():
    try:
        corp_actions_response = corporate_action_sources_api.get_instrument_events(scope = module_scope, code = module_code)
        corp_actions_response_df = lusid_response_to_data_frame(corp_actions_response, use_camel_case=True)
        display(corp_actions_response_df.transpose())        
    except ApiException as e:
        print(e)

check_CAS_event()

### Check events and output transactions again

`ProtectionPayoutCashFlowEvent` is emitted and a corresponding output transaction generated for a cash receivable on the event payment date, with subsequent premium payments adjusted down accordingly.

In [ ]:
# Just for AbovePar portfolio, since events are the same for both
print(ports[0])
query_instr_events(ports[0])

In [ ]:
# Just for AbovePar portfolio, since events are the same for both
print(ports[0])
get_output_transactions(ports[0], "2024-01-01", "2030-01-01", False)

### Examine holdings over the lifetime of the contract

In [ ]:
for port in ports:
    print(f"\n{port}")
    get_portfolio_holdings(port, "2024-09-25 23:59:59")  # Settlement date
    get_portfolio_holdings(port, "2024-12-20 23:59:59")  # After first premium payment
    get_portfolio_holdings(port, "2025-03-25 23:59:59")  # After protection payout on 25 March 2025 
    get_portfolio_holdings(port, "2030-01-01 23:59:59")  # After instrument maturity

## Valuation

See https://support.lusid.com/docs/modelling-cds-index-instruments-in-lusid#valuing-your-position

Random valuation date: "today"

### Load market data

The recommended `CdsLookupPricer` pricing model accepts a par-like price (eg. `102`) with a scale factor of `1`. 

In [ ]:
def load_quotes(luid, price, date, ccy, scale):
    if date == "today":
        date = datetime.today().strftime('%Y-%m-%d 00:00:00')

    quotes = {
        # Each quote must be upserted with an ephemeral key (uuid in this case), to track errors in the response
        str(uuid.uuid4()): lm.UpsertQuoteRequest(
            quote_id = lm.QuoteId(
                quote_series_id = lm.QuoteSeriesId(
                    # Must be one of the valid financial data vendor 'provider' values
                    provider = "Lusid",
                    instrument_id_type = "LusidInstrumentId",
                    instrument_id = luid,
                    quote_type = "Price",
                    # Case sensitive: the field value must match that of the equivalent recipe field exactly
                    field = "mid",
                ),
                effective_at = datetime.strptime(date, '%Y-%m-%d %H:%M:%S').replace(tzinfo=timezone.utc).isoformat(),
            ),
            metric_value = lm.MetricValue(value = price, unit = ccy),
            scale_factor = scale,
        )
    }

    try:
        upsert_quotes_response = quotes_api.upsert_quotes(scope = f"{module_scope}{module_code}", request_body = quotes)    
        if upsert_quotes_response.failed == {}:
            print(f"Price for {date} successfully loaded into LUSID.")
        else:
            print(f"Some failures occurred. {len(upsert_quotes_response.failed)} prices did not get loaded into LUSID.")
    except ApiException as e:
        print(e)

In [ ]:
load_quotes(luid_dict["MarkitiTraxxCDX"], 100, "2025-12-31 00:00:00", "EUR", 1) # Required for ProfitAndLoss keys with a YTD window
load_quotes(luid_dict["MarkitiTraxxCDX"], 100, "today", "EUR", 1) # Random valuation date

In [ ]:
def list_quotes():
    try:
        quotes_response = quotes_api.list_quotes_for_scope(f"{module_scope}{module_code}")
        quotes_response_df = lusid_response_to_data_frame(quotes_response)
        display(quotes_response_df)        
    except ApiException as e:
        print(e)

list_quotes()

### Perform valuation

In [ ]:
def value_instruments(port, date, pnl_window):
    if date == "today":
        date = datetime.today().strftime('%Y-%m-%d')

    valuation_request = lm.ValuationRequest(
        # Choose recipe to use
        recipe_id = lm.ResourceId(scope = module_scope, code = f"{module_code}-CdsLookupPricer"),
        # Specify metrics (also known as queryable keys) to report useful information
        metrics = [
            lm.AggregateSpec(key="Instrument/InstrumentCategory", op="Value"),
            lm.AggregateSpec(key="Instrument/default/Name", op="Value"),
            lm.AggregateSpec(key="Instrument/default/LusidInstrumentId", op="Value"),
            lm.AggregateSpec(key="Valuation/Model/Name", op="Value"),
            lm.AggregateSpec(key="Valuation/EffectiveAt", op="Value"),
            lm.AggregateSpec(key="Holding/default/Units", op="Value"),
            lm.AggregateSpec(key="Quotes/PriceOrFXRate", op="Value"),
            lm.AggregateSpec(key="Holding/Cost/Dom", op="Value"),
            lm.AggregateSpec(key="Valuation/CleanPV", op="Value"),
            lm.AggregateSpec(key="Valuation/PV", op="Value"),
            lm.AggregateSpec(key="Valuation/Accrued", op="Value"),
            # lm.AggregateSpec(key="Valuation/Exposure", op="Value"),          
            # lm.AggregateSpec(key="Valuation/CurrentNotional", op="Value"),
            lm.AggregateSpec(key="ProfitAndLoss/Total", op="Value", options={"Window": f"{pnl_window}"}),      
            lm.AggregateSpec(key="ProfitAndLoss/Total/Market", op="Value", options={"Window": f"{pnl_window}"}),
            lm.AggregateSpec(key="ProfitAndLoss/Realised/Market", op="Value", options={"Window": f"{pnl_window}"}),       
            lm.AggregateSpec(key="ProfitAndLoss/Unrealised/Market", op="Value", options={"Window": f"{pnl_window}"}),       
            lm.AggregateSpec(key="ProfitAndLoss/Total/Other", op="Value", options={"Window": f"{pnl_window}"}),
            lm.AggregateSpec(key="Aggregation/Errors", op="Value"), 
        ],
        # Identify portfolio to value
        portfolio_entity_ids = [lm.PortfolioEntityId(scope = module_scope, code = f"{module_code}-{port}")],
        valuation_schedule = lm.ValuationSchedule(effective_at = date),

    )

    try:
        # Get portfolio valuation
        val_response = aggregation_api.get_valuation(valuation_request = valuation_request)
        val_response_df = pd.json_normalize(val_response.to_dict()["data"], sep='.')
        # Rename columns
        val_response_df.rename(
            columns = {
                "Instrument/InstrumentCategory": "Category",
                "Valuation/Model/Name": "Pricing model",
                "Instrument/default/LusidInstrumentId": "LUID",
                "Instrument/default/Name": "Name",
                "Valuation/EffectiveAt": "Date",
                "Holding/default/Units": "Units",
                "Quotes/PriceOrFXRate": "Price",
                "Quotes/ScaleFactor": "Quote Scale Factor",
                "Holding/Cost/Dom": "Local Cost",
                "Valuation/CleanPV": "Local Clean PV",
                "Valuation/PV": "Local PV",
                "Valuation/Accrued": "Local Accrued Interest",
                "Valuation/Exposure": "Exposure",
                "Valuation/CurrentNotional": "Notional",            
                f"ProfitAndLoss/Total(Window=\"{pnl_window}\")": "Total P&L",
                f"ProfitAndLoss/Total/Market(Window=\"{pnl_window}\")": "Total/Market P&L",
                f"ProfitAndLoss/Realised/Market(Window=\"{pnl_window}\")": "Realised/Market P&L",
                f"ProfitAndLoss/Unrealised/Market(Window=\"{pnl_window}\")": "Unrealised/Market P&L",
                f"ProfitAndLoss/Total/Other(Window=\"{pnl_window}\")": "Total/Other P&L",
                "Aggregation/Errors": "Errors"
            },
            inplace = True,
        )
        val_response_df["Date"] = pd.to_datetime(val_response_df["Date"]).dt.date
        display(val_response_df)
    except ApiException as e:
        print(e)

In [ ]:
for port in ports:
    print(port)
    value_instruments(port, "today", "YTD") # Random valuation date

## Fund accounting

### Set up simple fund

In [ ]:
def create_coa():
    coa_request = lm.ChartOfAccountsRequest(
        code = module_code,
        display_name = f"{module_scope}/{module_code} CoA",
    )
    
    try:
        create_coa_response = chart_accts_api.create_chart_of_accounts(
            scope = module_scope,
            chart_of_accounts_request=coa_request
        )
        print("Success")
    except ApiException as e:
        print(e.body)
        
create_coa()

In [ ]:
def add_accounts():
    try:
        add_accounts_response = chart_accts_api.upsert_accounts(
           scope = module_scope,
            code = module_code,
            account = [
                lm.Account(
                    code = "1-Investments",
                    description="Investment account",
                    type = "Asset",
                    status = "Active",
                    control = "Manual",
                ),
                lm.Account(
                    code = "2-Cash",
                    description="Cash account",
                    type = "Liabilities",
                    status = "Active",
                    control = "Manual",
                ),
                lm.Account(
                    code = "3-Subscriptions",
                    description="Subscriptions account",
                    type = "Capital",
                    status = "Active",
                    control = "Manual",
                ),
                lm.Account(
                    code = "4-Redemptions",
                    description="Redemptions account",
                    type = "Capital",
                    status = "Active",
                    control = "Manual",
                ),
                lm.Account(
                    code = "5-PnL",
                    description="P&L account",
                    type = "Revenue",
                    status = "Active",
                    control = "Manual",
                ),
                lm.Account(
                    code = "6-YearEnd",
                    description="Cleardown account",
                    type = "Revenue",
                    status = "Active",
                    control = "Manual",
                ),
                lm.Account(
                    code = "7-Error",
                    description="Error account",
                    type = "Revenue",
                    status = "Active",
                    control = "Manual",
                ),
            ]
        )
        print("Success")
    except ApiException as e:
        print(e.body)

add_accounts()

In [ ]:
def examine_coa():
    
    coa_response = chart_accts_api.list_accounts(
        scope = module_scope, 
        code = module_code,
        # Retrieve properties to make results more intuitive
        #property_keys = ["Instrument/default/Name"],
    )

    try:
        coa_df = lusid_response_to_data_frame(coa_response)
        # Drop some noisy columns
        coa_df.drop(columns=["control"], inplace=True)
        display(coa_df)
    except ApiException as e:
        print(e.body)
        
examine_coa()

In [ ]:
def add_posting_module():
    pm_request = lm.PostingModuleRequest(
        code = module_code,
        display_name = f"{module_scope}/{module_code} posting module",
        rules = [
            lm.PostingModuleRule(
                rule_id = "rule_1",
                generalLedgerAccountCode = "3-Subscriptions",
                rule_filter = "EconomicBucket startswith 'CA' and Transaction.type eq 'FundsInWithCapitalMovement'"
            ),
            lm.PostingModuleRule(
                rule_id = "rule_2",
                generalLedgerAccountCode = "4-Redemptions",
                rule_filter = "EconomicBucket startswith 'CA' and Transaction.type eq 'FundsOutWithCapitalMovement'"
            ),
            lm.PostingModuleRule(
                rule_id = "rule_3",
                generalLedgerAccountCode = "1-Investments",
                rule_filter = "HoldType eq 'P' and EconomicBucket startswith 'NA'"
            ),
            # Assign tricksy P&L JE Lines to error account first
            lm.PostingModuleRule(
                rule_id = "rule_4",
                generalLedgerAccountCode = "7-Error",
                rule_filter = "EconomicBucket eq 'PL_Other' or EconomicBucket eq 'PL_Rounding'"
            ),
            # Assign legit P&L JE lines
            lm.PostingModuleRule(
                rule_id = "rule_5",
                generalLedgerAccountCode = "5-PnL",
                rule_filter = "EconomicBucket startswith 'PL'"
            ),
            lm.PostingModuleRule(
                rule_id = "rule_6",
                generalLedgerAccountCode = "2-Cash",
                rule_filter = "HoldType neq 'P' and EconomicBucket startswith 'NA'"
            ),
            # Assign unmatched JE lines to error account, if any
            lm.PostingModuleRule(
                rule_id = "rule_7",
                generalLedgerAccountCode = "7-Error",
                rule_filter = "True"
            )
        ]
    )
    
    try:
        pm_response = chart_accts_api.create_posting_module(
            scope = module_scope,
            code = module_code,
            posting_module_request=pm_request
        )
        print("Success")
    except ApiException as e:
        print(e.body)

add_posting_module()

In [ ]:
def examine_posting_module(code):
    
    pm_response = chart_accts_api.list_posting_module_rules(
        scope = module_scope, 
        code = module_code,
        posting_module_code = code
    )
    
    pm_response_df = lusid_response_to_data_frame(pm_response)
    display(pm_response_df)
    
examine_posting_module(module_code)

In [ ]:
# Create pricing template (has to exist before fund entity)
def create_fund_config():
    create_fundconfig_request = lm.FundConfigurationRequest(
        code = module_code,
        display_name = f"{module_scope}/{module_code} pricing template",
        dealingFilters = [
            lm.ComponentFilter(
                filterId="SUBS",
                filter="generalLedgerAccountCode eq '3-Subscriptions'"
            ),
            lm.ComponentFilter(
                filterId="REDS",
                filter="generalLedgerAccountCode eq '4-Redemptions'"
            ),
        ],
        pnlFilters=[
            lm.ComponentFilter(
                filterId="PNL",
                filter="generalLedgerAccountCode eq '5-PnL'"
            ),
        ],
        backOutFilters=[],
        # externalFeeFilters=[]
    )
    
    try:
        create_fundconfig_response = fundconfig_api.create_fund_configuration(
            scope = module_scope,
            fund_configuration_request=create_fundconfig_request
        )
        print("Success")
    except ApiException as e:
        print(e.body)

create_fund_config()

In [ ]:
def create_fund(ibor, ccy):
    
    create_fund_request = lm.FundDefinitionRequest(
        code = f"{module_code}-{ibor}",
        display_name = f"{module_code}-{ibor} fund",
        baseCurrency = ccy,
        portfolio_ids = [
            lm.PortfolioEntityId(
                scope = module_scope,
                code = f"{module_code}-{ibor}"
            )
        ],
        fundConfigurationId=lm.ResourceId(
            scope = module_scope,
            code = module_code
        ),
        investorStructure="NonUnitised",
        type = "Standalone",
        inceptionDate=to_date("2024-01-01"),  # Same as portfolios
        decimalPlaces=5,
        primaryNavType = lm.NavTypeDefinition(
            code = "OFFICIAL",
            displayName="Official NAV",
            description="This is the Official primary NAV type",
            valuationRecipeId=lm.ResourceId(
                scope = module_scope,
                code = f"{module_code}-CdsLookupPricer"
            ),
            holdingRecipeId=lm.ResourceId(
                scope = module_scope,
                code = f"{module_code}-CdsLookupPricer"
            ),
            accountingMethod = "AverageCost",
            amortisationMethod = "NoAmortisation",
            chart_of_accounts_id = lm.ResourceId(
                scope = module_scope,
                code = module_code
            ),
            posting_module_codes = [module_code],
            #cleardownModuleCodes=[module_code],
            cashGainLossCalculationDate="SettlementDate",
            transactionTypeScope=f"{module_scope}{module_code}",
            transactionTemplateScope="default"
        )
    )
    
    #Upsert to LUSID
    try:
        create_fund_response = fund_api.create_fund_v2(
            scope = module_scope,
            fund_definition_request=create_fund_request
        )
        #print(create_fund_response)
        print(f"Fund with display name '{create_fund_response.display_name}' created")
    except ApiException as e:
        print(e.body)

for port in ports:
    create_fund(port, "EUR")

### Examine JE Lines

In [ ]:
def get_je_lines(ibor, dateordiary, show_summary):
    if dateordiary == "today":
        dateordiary = datetime.today().strftime('%Y-%m-%dT23:59:59Z')
    
    try:
        jelines_response = fund_api.get_valuation_point_journal_entry_lines(
            scope = module_scope,
            code = f"{module_code}-{ibor}",
            valuation_point_data_query_parameters = lm.ValuationPointDataQueryParameters(end=lm.DateOrDiaryEntry(date=dateordiary)),
            nav_type_code="OFFICIAL"
            #filter = "sourceType eq 'LusidTransaction'"
        )
        jelines_response_df = lusid_response_to_data_frame(jelines_response)
        jelines_response_df.sort_values(by=['accounting_date'], inplace=True)
        if show_summary == True:
            jelines_response_df[["accounting_date", "instrument_id", "general_ledger_account_code", 
                                 "base.amount", "source_type", "movement_name", "holding_type", "economic_bucket", "ledger_column"]]
            jelines_response_df = jelines_response_df.filter(items=[
                "accounting_date",
                "instrument_id",
                "general_ledger_account_code",
                "base.amount",
                "source_type",
                "movement_name",
                "holding_type",
                "economic_bucket",
                "ledger_column"
            ])
            display(jelines_response_df)
        else:
            display(jelines_response_df)
    except ApiException as e:
        print(e)

In [ ]:
for port in ports:
    print(f"\n{port} fund:\n")
    get_je_lines(port, "today", True)
   # get_je_lines(port, "today", False)

### Examine trial balance

In [ ]:
def get_tb(ibor, dateordiary):
    if dateordiary == "today":
        dateordiary = datetime.today().strftime('%Y-%m-%dT23:59:59Z')
        
    try:
        tb_response = fund_api.get_valuation_point_trial_balance(
            scope = module_scope,
            code = f"{module_code}-{ibor}",
            valuation_point_data_query_parameters = lm.ValuationPointDataQueryParameters(end=lm.DateOrDiaryEntry(date=dateordiary)),
            nav_type_code="OFFICIAL"
        )

        #print(tb_response)
        tb_df = pd.json_normalize(tb_response.to_dict()["values"], sep='.').fillna('')
        tb_df.drop(columns=["levels", "localCurrency", "opening.localAmount", "closing.localAmount", "debit.localAmount", "credit.localAmount"], inplace=True)
        tb_df.rename(columns={'generalLedgerAccountCode': "account",
                              'opening.baseAmount': 'open', 
                              'closing.baseAmount': 'close',
                              'debit.baseAmount': 'DR', 
                              'credit.baseAmount': 'CR'
                             }, inplace=True)

        tb_df = (
                tb_df.assign(
                  order=1,
                  section=np.where(tb_df.accountType.isin({'Asset','Liabilities'}),1,2)
                  )
                  .pipe(lambda tb_df : pd.concat([
                      tb_df,
#                      tb_df.groupby('section',as_index=False)
#                         .sum(numeric_only=True)
#                         .assign(account='Subtotal',
#                                 order=2),
                      tb_df.sum(numeric_only=True)
                        .to_frame()
                        .T
                        .assign(account='Grand Total',
                                order=3)
                      ]))
                  .fillna('')
                  .sort_values(['section','order','account'])
                  .drop(['section','order'],axis=1)
             )

        # # #tb_df = (tb_df.set_index(['general_ledger_account_code','description', 'account_type', 'level 1', 'level 2', 'level 3', 'level 4'])[['opening', 'debit', 'credit', 'closing']])
        tb_df = (tb_df.set_index(['account','description', 'accountType'])[['open','DR','CR', 'close']])
        display(tb_df)
    except ApiException as e:
        print(e.body)

In [ ]:
for port in ports:
    print(f"\n{port} fund:")
    get_tb(port, "today")